# TC3-38 – Levantar e integrar bases socioeconômicas e financeiras municipais

**Responsável:** Mauricio Cruz

**Sprint:** Fase 2

**Jira:** TC3-82

## Objetivo

Levantar, baixar, documentar e integrar as novas bases socioeconômicas e financeiras municipais que podem enriquecer as análises exploratórias e futuras etapas de modelagem.



### Carrega dados ml-alunos utilizando os passos do TC3_38

In [63]:


import pandas as pd
import zipfile
from pathlib import Path

# Localiza a raiz do projeto a partir do diretório atual do notebook.
# Assim o notebook funciona no VS Code, Jupyter local ou outro clone do repositório.
def encontrar_raiz_projeto():
    atual = Path.cwd().resolve()
    for candidato in [atual, *atual.parents]:
        if (candidato / "dados" / "principal" / "ml_aluno.zip").exists():
            return candidato
    raise FileNotFoundError(
        "Não foi possível encontrar dados/principal/ml_aluno.zip. "
        "Execute o notebook dentro do repositório ou confira se os dados foram baixados."
    )

PROJECT_ROOT = encontrar_raiz_projeto()
ZIP_PATH = PROJECT_ROOT / "dados" / "principal" / "ml_aluno.zip"
EXTRACT_PATH = PROJECT_ROOT / "dados" / "principal"

if not (EXTRACT_PATH / "ml_aluno").exists():
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)

# Caminho local da feature table extraída
BASE_PATH = EXTRACT_PATH / "ml_aluno"

# Partições da Gold
PATH_2023 = BASE_PATH / "ano2023"
PATH_2024 = BASE_PATH / "ano2024"

arquivos_2023 = sorted(PATH_2023.glob("*.parquet"))
arquivos_2024 = sorted(PATH_2024.glob("*.parquet"))

df_2023 = pd.concat(
    [pd.read_parquet(arq) for arq in arquivos_2023],
    ignore_index=True
)

df_2024 = pd.concat(
    [pd.read_parquet(arq) for arq in arquivos_2024],
    ignore_index=True
)

# Consolidação da base
df = pd.concat([df_2023, df_2024], ignore_index=True)

print("=" * 40)
print("FEATURE TABLE CONSOLIDADA")
print("=" * 40)
print(f"Registros: {len(df):,}")
print(f"Colunas: {df.shape[1]}")

FEATURE TABLE CONSOLIDADA
Registros: 3,867,999
Colunas: 19


### Carrega arquivos complementares

In [77]:


PATH_COMPLEMENTAR = PROJECT_ROOT / "dados" / "complementar" 

#Bolsa Familia
df_bolsa_bronze = pd.read_csv(PATH_COMPLEMENTAR / "base_dados_bolsa_familia_2023_2024.csv")

#Fundeb
df_fundeb_bronze = pd.read_csv(PATH_COMPLEMENTAR / "base_dados_fundeb_indicador_municipal_2023_2024.csv")

#Receitas Orcamentarias
df_receitas_bronze = pd.read_csv(PATH_COMPLEMENTAR / "base_de_dados_municipio_receitas_orcamentarias_2023_2024.csv")

#PIB
df_pib_bronze = pd.read_csv(
    PATH_COMPLEMENTAR / "base_de_dados_PIB_Municipios_2022_2023.csv",
    sep=";",           # o arquivo usa ;
    decimal=","        # números com vírgula
)

#INSE
df_inse_bronze = pd.read_excel(PATH_COMPLEMENTAR / "INSE_2023_escolas.xlsx", sheet_name="INSE_ESC_2023")  # ou sheet_name=0 (primeira aba)


#Censo Escolar
ZIP_PATH_CENSO = PATH_COMPLEMENTAR / "microdados_ed_basica_2023.zip"
CSV_PATH_CENSO = PATH_COMPLEMENTAR / "microdados_ed_basica_2023.csv"

if ZIP_PATH_CENSO.exists() and not CSV_PATH_CENSO.exists():
    with zipfile.ZipFile(ZIP_PATH_CENSO, "r") as zip_ref:
        zip_ref.extractall(PATH_COMPLEMENTAR)

if CSV_PATH_CENSO.exists():
    df_censo_bronze = pd.read_csv(
        CSV_PATH_CENSO,
        sep=";",
        decimal=",",
        encoding="latin1"
    )
else:
    print("Arquivo do censo não encontrado.")








C:\Users\Mauripi\AppData\Local\Temp\ipykernel_960\3810150610.py:32: DtypeWarning: Columns (31: CO_ORGAO_REGIONAL) have mixed types. Specify dtype option on import or set low_memory=False.
  df_censo_bronze = pd.read_csv(


### Padronizar os nomes das colunas

In [113]:
# Bolsa Família
df_bolsa_silver = (
    df_bolsa_bronze
    .rename(columns={
        "ano_referencia": "ano",
        "total_valor": "valor_bolsa_familia"
    })
    .rename(columns=lambda col: col.lower())
)

# Fundeb
df_fundeb_silver = (
    df_fundeb_bronze
    .rename(columns={
        "total_valor": "valor_fundeb"
    })
    .rename(columns=lambda col: col.lower())
)

# Receitas orçamentárias
df_receitas_silver = (
    df_receitas_bronze
    .rename(columns={
        "total_valor": "valor_receitas"
    })
    .rename(columns=lambda col: col.lower())
)

# PIB
df_pib_silver = (
    df_pib_bronze
    .rename(columns={
        "Ano": "ano",
        "UF": "sigla_uf",
        "Código do Município": "id_municipio",
        "Produto Interno Bruto, a preços correntes": "valor_pib",
        "Produto Interno Bruto per capita, a preços correntes": "valor_pib_per_capita"
    })
    .rename(columns=lambda col: col.lower())
)

# INSE
df_inse_silver = (
    df_inse_bronze
    .rename(columns={
        "NU_ANO_SAEB": "ano",
        "SG_UF": "sigla_uf",
        "NO_UF": "nome_uf",
        "CO_MUNICIPIO": "id_municipio",
        "NO_MUNICIPIO": "nome_municipio",
        "ID_ESCOLA": "id_escola",
        "NO_ESCOLA": "nome_escola",
        "TP_TIPO_REDE": "tipo_rede",
        "TP_LOCALIZACAO": "localizacao",
        "TP_CAPITAL": "capital",
        "QTD_ALUNOS_INSE": "qtd_alunos_inse",
        "MEDIA_INSE": "media_inse",
        "INSE_CLASSIFICACAO": "inse_classificacao",
        "PC_NIVEL_1": "pc_nivel_1",
        "PC_NIVEL_2": "pc_nivel_2",
        "PC_NIVEL_3": "pc_nivel_3",
        "PC_NIVEL_4": "pc_nivel_4",
        "PC_NIVEL_5": "pc_nivel_5",
        "PC_NIVEL_6": "pc_nivel_6",
        "PC_NIVEL_7": "pc_nivel_7",
        "PC_NIVEL_8": "pc_nivel_8"
    })
    .rename(columns=lambda col: col.lower())
)

# Censo
df_censo_silver = (
    df_censo_bronze
    .rename(columns={
        "NU_ANO_CENSO": "ano",
        "SG_UF": "sigla_uf",
        "NO_UF": "nome_uf",
        "CO_UF": "co_uf",
        "CO_MUNICIPIO": "id_municipio",
        "NO_MUNICIPIO": "nome_municipio",
        "CO_ENTIDADE": "id_escola",
        "NO_ENTIDADE": "nome_escola",
        "NO_REGIAO": "nome_regiao",
        "CO_REGIAO": "co_regiao",
        "TP_LOCALIZACAO": "localizacao",
        "TP_LOCALIZACAO_DIFERENCIADA": "localizacao_diferenciada",
        "TP_DEPENDENCIA": "dependencia",
        "IN_BIBLIOTECA_SALA_LEITURA": "tem_biblioteca",
        "IN_LABORATORIO_INFORMATICA": "tem_lab_informatica",
        "IN_QUADRA_ESPORTES": "tem_quadra",
        "IN_AGUA_REDE_PUBLICA": "tem_agua_rede_publica",
        "IN_ENERGIA_REDE_PUBLICA": "tem_energia_rede_publica",
        "IN_ESGOTO_REDE_PUBLICA": "tem_esgoto_rede_publica",
        "TP_SITUACAO_FUNCIONAMENTO": "situacao_funcionamento",
        "IN_INTERNET": "tem_internet",
        "IN_BANDA_LARGA": "tem_banda_larga",
        "QT_MAT_BAS": "qtd_matriculas_basica",
        "QT_DOC_BAS": "qtd_docentes_basica",
        "QT_TUR_BAS": "qtd_turmas_basica"
    })
    .rename(columns=lambda col: col.lower())
)


### Padronizar os tipos das colunas

In [114]:
df["id_municipio"] = df["id_municipio"].astype("int64")
df["ano_referencia"] = df["ano_referencia"].astype("int64")
df_aluno_silver = df

df_bolsa_silver["valor_bolsa_familia"] = df_bolsa_silver["valor_bolsa_familia"].astype("float64")
df_fundeb_silver["valor_fundeb"] = df_fundeb_silver["valor_fundeb"].astype("float64")
df_receitas_silver["valor_receitas"] = df_receitas_silver["valor_receitas"].astype("float64")
df_pib_silver["valor_pib"] = (
    df_pib_silver["valor_pib"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
    .astype(float)
)
df_pib_silver["valor_pib_per_capita"] = (
    df_pib_silver["valor_pib_per_capita"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
    .astype(float)
)

### Unir todas as bases

#### Como o dataset principal (df_aluno) tem vários registros por município/ano, é realizado o left join a partir dele

In [115]:
# Junta Bolsa Família
df_aluno_silver = df_aluno_silver.merge(
    df_bolsa_silver,
    how="left",
    left_on=["ano_referencia", "id_municipio", "sigla_uf"],
    right_on=["ano", "id_municipio", "sigla_uf"]
)

# Junta Fundeb
df_aluno_silver = df_aluno_silver.merge(
    df_fundeb_silver,
    how="left",
    left_on=["ano_referencia", "id_municipio", "sigla_uf"],
    right_on=["ano", "id_municipio", "sigla_uf"],
    suffixes=("", "_fundeb")   # evita conflito de nomes
)

# Junta Receitas
df_aluno_silver = df_aluno_silver.merge(
    df_receitas_silver,
    how="left",
    left_on=["ano_referencia", "id_municipio", "sigla_uf"],
    right_on=["ano", "id_municipio", "sigla_uf"],
    suffixes=("", "_receitas")
)

# Junta PIB
df_aluno_silver = df_aluno_silver.merge(
    df_pib_silver,
    how="left",
    left_on=["ano_referencia", "id_municipio", "sigla_uf"],
    right_on=["ano", "id_municipio", "sigla_uf"]
)

#Junta INSE
df_aluno_silver = df_aluno_silver.merge(
    df_inse_silver,
    how="left",
    left_on=["ano_referencia", "id_municipio", "sigla_uf"],
    right_on=["ano", "id_municipio", "sigla_uf"]
)

df_aluno_silver = df_aluno_silver.merge(
    df_censo_silver,
    how="left",
    left_on=["ano_referencia", "id_municipio", "sigla_uf"],
    right_on=["ano", "id_municipio", "sigla_uf"]
)



ArrowMemoryError: realloc of size 4390595584 failed

In [109]:
df_aluno_silver.info()
df_aluno_silver.head()

<class 'pandas.DataFrame'>
RangeIndex: 3867999 entries, 0 to 3867998
Data columns (total 30 columns):
 #   Column                                     Dtype         
---  ------                                     -----         
 0   id_aluno                                   str           
 1   id_escola                                  str           
 2   id_municipio                               int64         
 3   nome_municipio                             str           
 4   sigla_uf                                   str           
 5   ano_referencia                             int64         
 6   serie                                      int32         
 7   rede                                       int32         
 8   rede_descricao                             str           
 9   caderno                                    str           
 10  presenca                                   int32         
 11  preenchimento_caderno                      int32         
 12  proficienci

,id_aluno,id_escola,id_municipio,nome_municipio,sigla_uf,ano_referencia,serie,rede,rede_descricao,caderno,...,valor_bolsa_familia,ano_fundeb,valor_fundeb,ano_receitas,valor_receitas,ano_y,nome uf,nome do município,valor_pib,valor_pib_per_capita
0,31062133,60021550,3136900,Juruaia,MG,2023,2,3,Municipal,6,...,2312976.0,2023.0,4.004472e+07,2023.0,4.986735e+08,2023.0,Minas Gerais,Juruaia,3.033282e+08,27366.31
1,15078120,60001463,1506005,Prainha,PA,2023,2,3,Municipal,6,...,57458393.0,2023.0,2.230224e+08,NaN,NaN,2023.0,Pará,Prainha,5.488861e+08,15428.12
2,43090743,60037407,4305108,Caxias do Sul,RS,2023,2,3,Municipal,6,...,87984651.0,2023.0,1.401397e+09,2023.0,2.527093e+10,2023.0,Rio Grande do Sul,Caxias do Sul,3.786060e+10,81683.97
3,42097998,60035930,4219507,Xanxerê,SC,2023,2,3,Municipal,6,...,6345159.0,2023.0,2.161164e+08,2023.0,2.137319e+09,2023.0,Santa Catarina,Xanxerê,3.145565e+09,60952.29
4,43072338,60037775,4309209,Gravataí,RS,2023,2,3,Municipal,7,...,110156423.0,2023.0,7.174176e+08,2023.0,1.051835e+10,2023.0,Rio Grande do Sul,Gravataí,1.554556e+10,58646.11
